In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import random
random.seed(9001)

In [3]:
from PyPOTS_gp_benchmark_helpers import artifical_GP_generation

dataset_train_nan, dataset_val_nan, dataset_train, dataset_val, X_normalized, X = artifical_GP_generation(n_observations_per_group = 500,
                                                n_dimensions = 3,
                                                n_time_points = 30,
                                                n_new_dims = 7,
                                                p_dataset = 0.5)

100%|█████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 3445.73it/s]


In [4]:
import sys
sys.path.append("..")

In [6]:

# Model training. This is PyPOTS showtime.
from pypots.imputation import GP_UAE
from pypots.utils.metrics import calc_mae
from pypots.optim.adam import Adam
from torch.optim.lr_scheduler import LRScheduler

gpuae = GP_UAE(n_steps = dataset_train['X'].shape[1], 
            n_features = dataset_train['X'].shape[2], 
            latent_size = 6, 
            epochs = 2000, 
            batch_size = 64,
            alpha = .9,
            beta = .01, 
            sigma = 1e-2,
            K = 1,  
            encoder_sizes = (32,64,), 
            decoder_sizes = (64,32,),
            optimizer = Adam(weight_decay=0.001), #lr = 1e-4
            patience = 100,
            p = 0.3
            )

gpuae.use_gp = False
gpuae.train_gp = False
gpuae.gp.plot_while_training = True
gpuae.model.backbone.device = 'cpu'
gpuae.model.backbone.encoder.pre_impute = False
gpuae.model.backbone.DAE = False
gpuae.model.backbone.kl_weight = 1

import torch
with torch.autograd.set_detect_anomaly(True):
    gpuae.fit(dataset_train_nan, dataset_val_nan)  # train the model on the dataset

2025-06-23 18:56:33 [INFO]: No given device, using default device: cpu
2025-06-23 18:56:33 [WARNING]: ‼️ saving_path not given. Model files and tensorboard file will not be saved.
2025-06-23 18:56:33 [INFO]: GP_UAE initialized with the given hyperparameters, the number of trainable parameters: 6,131



████████╗██╗███╗   ███╗███████╗    ███████╗███████╗██████╗ ██╗███████╗███████╗    █████╗ ██╗
╚══██╔══╝██║████╗ ████║██╔════╝    ██╔════╝██╔════╝██╔══██╗██║██╔════╝██╔════╝   ██╔══██╗██║
   ██║   ██║██╔████╔██║█████╗█████╗███████╗█████╗  ██████╔╝██║█████╗  ███████╗   ███████║██║
   ██║   ██║██║╚██╔╝██║██╔══╝╚════╝╚════██║██╔══╝  ██╔══██╗██║██╔══╝  ╚════██║   ██╔══██║██║
   ██║   ██║██║ ╚═╝ ██║███████╗    ███████║███████╗██║  ██║██║███████╗███████║██╗██║  ██║██║
   ╚═╝   ╚═╝╚═╝     ╚═╝╚══════╝    ╚══════╝╚══════╝╚═╝  ╚═╝╚═╝╚══════╝╚══════╝╚═╝╚═╝  ╚═╝╚═╝
ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 

Model dimensions is:
GpvaeEncoder(
  (net): Sequential(
    (0): Linear(in_features=14, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
  )
  (mu_layer): Linear(in_features=64, out_features=6, bias=True)
  (logvar_layer): Linear(in_features=64, out_features=6

2025-06-23 18:56:33 [INFO]: Epoch 001 - training loss: 47.2210, validation loss: 51.5062
2025-06-23 18:56:33 [INFO]: Epoch 002 - training loss: 47.5869, validation loss: 51.1496
2025-06-23 18:56:33 [INFO]: Epoch 003 - training loss: 47.4598, validation loss: 51.0582
2025-06-23 18:56:34 [INFO]: Epoch 004 - training loss: 46.8627, validation loss: 50.3160
2025-06-23 18:56:34 [INFO]: Epoch 005 - training loss: 45.5507, validation loss: 48.1188
2025-06-23 18:56:35 [INFO]: Epoch 006 - training loss: 43.9323, validation loss: 45.0737
2025-06-23 18:56:35 [INFO]: Epoch 007 - training loss: 38.3357, validation loss: 37.6792
2025-06-23 18:56:35 [INFO]: Epoch 008 - training loss: 31.7123, validation loss: 27.8454
2025-06-23 18:56:35 [INFO]: Epoch 009 - training loss: 24.7271, validation loss: 23.1916
2025-06-23 18:56:36 [INFO]: Epoch 010 - training loss: 21.7506, validation loss: 20.9436
2025-06-23 18:56:36 [INFO]: Epoch 011 - training loss: 18.4534, validation loss: 17.9000
2025-06-23 18:56:36 [

Plotting latent series and reconstruction...


2025-06-23 18:56:46 [INFO]: Epoch 056 - training loss: 4.9705, validation loss: 4.8876
2025-06-23 18:56:46 [INFO]: Epoch 057 - training loss: 4.8160, validation loss: 5.0953
2025-06-23 18:56:46 [INFO]: Epoch 058 - training loss: 4.9486, validation loss: 4.9409
2025-06-23 18:56:46 [INFO]: Epoch 059 - training loss: 4.6695, validation loss: 4.9089
2025-06-23 18:56:47 [INFO]: Epoch 060 - training loss: 4.6768, validation loss: 4.8208
2025-06-23 18:56:47 [INFO]: Epoch 061 - training loss: 4.6371, validation loss: 4.7843
2025-06-23 18:56:47 [INFO]: Epoch 062 - training loss: 4.5698, validation loss: 4.7128
2025-06-23 18:56:47 [INFO]: Epoch 063 - training loss: 4.5258, validation loss: 4.5093
2025-06-23 18:56:47 [INFO]: Epoch 064 - training loss: 4.3910, validation loss: 4.5481
2025-06-23 18:56:48 [INFO]: Epoch 065 - training loss: 4.3225, validation loss: 4.6192
2025-06-23 18:56:48 [INFO]: Epoch 066 - training loss: 4.3176, validation loss: 4.3328
2025-06-23 18:56:48 [INFO]: Epoch 067 - tra

...

2025-06-23 18:59:01 [INFO]: Epoch 667 - training loss: 0.4435, validation loss: 0.3488
2025-06-23 18:59:01 [INFO]: Epoch 668 - training loss: 0.4314, validation loss: 0.4059
2025-06-23 18:59:01 [INFO]: Epoch 669 - training loss: 0.4601, validation loss: 0.3938
2025-06-23 18:59:01 [INFO]: Epoch 670 - training loss: 0.4628, validation loss: 0.3835
2025-06-23 18:59:01 [INFO]: Epoch 671 - training loss: 0.4607, validation loss: 0.3463
2025-06-23 18:59:02 [INFO]: Epoch 672 - training loss: 0.4386, validation loss: 0.3782
2025-06-23 18:59:02 [INFO]: Epoch 673 - training loss: 0.4375, validation loss: 0.3812
2025-06-23 18:59:02 [INFO]: Epoch 674 - training loss: 0.4637, validation loss: 0.4025
2025-06-23 18:59:02 [INFO]: Epoch 675 - training loss: 0.4542, validation loss: 0.3607
2025-06-23 18:59:03 [INFO]: Epoch 676 - training loss: 0.4543, validation loss: 0.3675
2025-06-23 18:59:03 [INFO]: Epoch 677 - training loss: 0.4568, validation loss: 0.3633
2025-06-23 18:59:03 [INFO]: Epoch 678 - tra

Plotting latent series and reconstruction...


2025-06-23 18:59:13 [INFO]: Epoch 723 - training loss: 0.4351, validation loss: 0.3322
2025-06-23 18:59:13 [INFO]: Epoch 724 - training loss: 0.4078, validation loss: 0.3517
2025-06-23 18:59:13 [INFO]: Epoch 725 - training loss: 0.4055, validation loss: 0.3343
2025-06-23 18:59:13 [INFO]: Epoch 726 - training loss: 0.4167, validation loss: 0.3639
2025-06-23 18:59:14 [INFO]: Epoch 727 - training loss: 0.4195, validation loss: 0.3654
2025-06-23 18:59:14 [INFO]: Epoch 728 - training loss: 0.4215, validation loss: 0.3219
2025-06-23 18:59:14 [INFO]: Epoch 729 - training loss: 0.4301, validation loss: 0.3527
2025-06-23 18:59:14 [INFO]: Epoch 730 - training loss: 0.4237, validation loss: 0.3294
2025-06-23 18:59:14 [INFO]: Epoch 731 - training loss: 0.4356, validation loss: 0.3445
2025-06-23 18:59:15 [INFO]: Epoch 732 - training loss: 0.3966, validation loss: 0.3557
2025-06-23 18:59:15 [INFO]: Epoch 733 - training loss: 0.4279, validation loss: 0.3292
2025-06-23 18:59:15 [INFO]: Epoch 734 - tra

Plotting latent series and reconstruction...


2025-06-23 18:59:25 [INFO]: Epoch 779 - training loss: 0.3932, validation loss: 0.3366
2025-06-23 18:59:25 [INFO]: Epoch 780 - training loss: 0.4288, validation loss: 0.3934
2025-06-23 18:59:25 [INFO]: Epoch 781 - training loss: 0.4012, validation loss: 0.3425
2025-06-23 18:59:25 [INFO]: Epoch 782 - training loss: 0.4116, validation loss: 0.3476
2025-06-23 18:59:25 [INFO]: Epoch 783 - training loss: 0.4136, validation loss: 0.3208
2025-06-23 18:59:26 [INFO]: Epoch 784 - training loss: 0.3875, validation loss: 0.3458
2025-06-23 18:59:26 [INFO]: Epoch 785 - training loss: 0.4133, validation loss: 0.3221
2025-06-23 18:59:26 [INFO]: Epoch 786 - training loss: 0.4161, validation loss: 0.3310
2025-06-23 18:59:26 [INFO]: Epoch 787 - training loss: 0.3966, validation loss: 0.3306
2025-06-23 18:59:26 [INFO]: Epoch 788 - training loss: 0.4002, validation loss: 0.3647
2025-06-23 18:59:27 [INFO]: Epoch 789 - training loss: 0.4016, validation loss: 0.3294
2025-06-23 18:59:27 [INFO]: Epoch 790 - tra

In [ ]:
#gpuae.model.backbone.alpha = 0.1
gpuae.fit(dataset_train_nan, dataset_val_nan)  # train the model on the dataset

In [ ]:
gpuae.model.backbone.alpha = 1e0
gpuae.model.gp.use_quantile = True
gpuae.model.gp.alpha = .8
gpuae.model.alpha = .8


gpuae.fit(dataset_train_nan, dataset_val_nan, train_kernel_only = True)  # train the model on the dataset

## Plot latent RPZ

In [ ]:
def plot_multi_gaussian(q, i = 0, alpha = 1.96, linestyle = '-'):
    """
    Plot all dimensions of the ith observation
    """
    
    mu, var = np.array(q.mean.detach().numpy())[i], np.array(q.variance.detach().numpy())[i]**.5

    plt.plot(mu, linestyle = linestyle);
   
    plt.gca().set_prop_cycle(None)
    for j in range(var.shape[-1]):
        low, up = (mu - alpha*var)[:,j], (mu + alpha*var)[:,j]
        plt.fill_between(np.arange(mu.shape[0]), low, up, alpha = .2);

def plot_reconstructions(X, X_ori, q, model, i = 0, n_samples = 100, ax = None):
    """
    Given the observed data X, the latent distribution q and the model (for the decoder) plots the reconstructions
    """

    z = q.rsample((n_samples,))
    #print(z[0,0,0])
    #print(z[1,0,0])
    #print('**')
    #z = q.mean.repeat((100,1,1,1))

    #print(z.shape, X.shape)

    X_recon = np.array(model.model.backbone.decoder(z).mean.detach().numpy())[:,i]

    if ax is None:
        fig, ax = plt.subplots(figsize=(18,8))

    # plot X
    X = np.array(X.detach().numpy())
    X_ori = np.array(X_ori.detach().numpy())

    X[X==0] = np.nan
    ax.plot(X[i], 'o');
    ax.set_prop_cycle(None)
    ax.plot(X_ori[i], '+');
    ax.set_prop_cycle(None)

    #for j in range(n_samples):
    #    ax.plot(X_recon[j], alpha = .05)
    #    ax.set_prop_cycle(None)
    low, up = X_recon.min(axis=0), X_recon.max(axis=0)
    for j in range(X_recon.shape[-1]):
        ax.fill_between(np.arange(X.shape[1]), low[:,j], up[:,j], alpha = .3)

    ax.set_prop_cycle(None)
    ax.plot(X_recon.mean(axis=0))

    scale_factor = 1.5
    ymin, ymax = np.nanmin(X[i])*scale_factor, np.nanmax(X[i])*scale_factor
    ax.set_ylim([ymin,ymax])
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

i = 3
alpha = .5

X_ori = torch.tensor(dataset_val_nan['X_ori']).float()

X_corr = torch.tensor(dataset_val_nan['X']).float()
X_corr[X_corr!=X_corr] = 0.

qz_x_corr = gpuae.model.backbone.encoder(X_corr)

# correct
kernel_params = gpuae.gp.update_kernel_params(X_corr)
qz_star = gpuae.gp.correct_qz_x_with_gp(qz_x_corr, kernel_params)

# plot reconstructions

kernel_params = gpuae.gp.update_kernel_params(X_corr)
qz_star = gpuae.gp.correct_qz_x_with_gp(qz_x_corr, kernel_params)

plot_multi_gaussian(qz_x_corr, alpha = 1)
plot_multi_gaussian(qz_star, alpha = 1, linestyle = ':')
plt.show();

fig, ax = plt.subplots(2, figsize=(18,8))

i = 4
plot_reconstructions(X_corr, X_ori, qz_star, gpuae, ax = ax[1], i = i)
plot_reconstructions(X_corr, X_ori, qz_x_corr, gpuae, ax = ax[0], i = i)


In [ ]:
# plot reconstructions

kernel_params = gpuae.gp.update_kernel_params(X_corr)
qz_star = gpuae.gp.correct_qz_x_with_gp(qz_x_corr, kernel_params)

plot_multi_gaussian(qz_x_corr, alpha = 1)
plot_multi_gaussian(qz_star, alpha = 1, linestyle = ':')
plt.show();

fig, ax = plt.subplots(2, figsize=(18,8))

i = 4
plot_reconstructions(X_corr, X_ori, qz_star, gpuae, ax = ax[1], i = i)
plot_reconstructions(X_corr, X_ori, qz_x_corr, gpuae, ax = ax[0], i = i)


In [ ]:
latent_variances = qz_x_corr.variance.reshape(-1,qz_x_corr.variance.shape[-1])
torch.quantile(latent_variances,q=0.1,dim=0)

In [ ]:
import torch
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.Sigmoid()        
            )

    def forward(self, x, q = 1e-5):
        x_shape = x.shape
        x = x.reshape(-1,self.latent_dim)
        mask_below_quantile = (x<=q)
        logits = torch.zeros(x.shape)
        logits[~mask_below_quantile] = self.linear_relu_stack(x[~mask_below_quantile])
        #logits = self.linear_relu_stack(x)
        #logits[mask_below_quantile] = logits[mask_below_quantile] * 0
        logits = logits.reshape(x_shape)
        return logits.clamp(max = 1)

class OneDimensionNeuralNetwork(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU()        
            )

    def forward(self, x, q = 1e-5):
        x_shape = x.shape
        x = x.reshape(-1,self.latent_dim)
        mask_above_quantile = (x>q)[:,0]
        logits = torch.zeros(x.shape) + 1e-6
        logits[mask_above_quantile] = self.linear_relu_stack(x[mask_above_quantile]).clamp(min = 1e-5, max = 10)
        #logits = self.linear_relu_stack(x)
        #logits[mask_below_quantile] = logits[mask_below_quantile] * 0
        logits = logits.reshape(x_shape)
        return logits

class VarianceCorrection(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.nns = [OneDimensionNeuralNetwork(1) for i in range(latent_dim)]

    def forward(self, x):
        x_shape = x.shape
        x = x.reshape(-1,self.latent_dim) #/ 1e-6
        logits = torch.vstack([self.nns[i](x[:,i]) for i in range(self.latent_dim)])
        logits = logits.reshape(x_shape)
        return logits.clamp(max = 1)
#gpuae.model.gp.nn = NeuralNetwork(latent_dim = 6)
gpuae.model.gp.nn = VarianceCorrection(latent_dim = 6)

In [ ]:
for i in range(qz_x_corr.variance.shape[-1]):
    plt.title('Histogram of variances outputed by a UAE for a given latent dimension')
    plt.hist(qz_x_corr.variance[:,:,i].flatten().detach().numpy(), bins = 50)
    plt.show();

In [ ]:
torch.quantile(qz_x_corr.variance[:,:,0].flatten(), q = 0.5)

In [ ]:
## learn_quantiles:

X_corr = torch.tensor(dataset_train_nan['X']).float()
X_corr[X_corr!=X_corr] = 0.

qz_x_corr = gpuae.model.backbone.encoder(X_corr)
z_var = qz_x_corr.variance

In [ ]:
plt.hist(z_var_reshaped[:,4].detach().numpy(), bins = 100);

In [ ]:
latent_dim = 6
z_var_reshaped = z_var.reshape(-1,latent_dim)
z_var_quantiles = torch.quantile(z_var_reshaped, q = 0.1, interpolation = 'higher', dim = 0)[None]
print(z_var_reshaped.shape)
z_var_reshaped = (z_var_reshaped >= z_var_quantiles).float()
print(z_var_reshaped.mean(axis=0))
latent_variances = z_var_reshaped.reshape(z_var.shape)

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import norm
from scipy.optimize import fsolve

# Simulated z_var for demonstration (replace this with actual input)
#n_obs, n_time_steps, n_latent_dims = 100, 50, 8
#np.random.seed(0)
#z_var = np.abs(np.random.randn(n_obs, n_time_steps, n_latent_dims)) * 0.1

# Function to compute thresholds per latent dimension using GMM
def compute_gmm_thresholds(z_var):
    n_latent_dims = z_var.shape[2]
    thresholds = []

    for dim in range(n_latent_dims):
        variances = z_var[:, :, dim].reshape(-1, 1).detach().numpy()  # Flatten to (n_samples, 1)
        
        # Fit GMM
        gmm = GaussianMixture(n_components=2, random_state=0).fit(variances)
        means = gmm.means_.flatten()
        covs = gmm.covariances_.flatten()
        weights = gmm.weights_.flatten()

        # Solve for intersection
        def diff(x):
            return (weights[0] * norm.pdf(x, means[0], np.sqrt(covs[0])) -
                    weights[1] * norm.pdf(x, means[1], np.sqrt(covs[1])))

        try:
            initial_guess = np.mean(means)
            tau = fsolve(diff, x0=initial_guess)[0]
        except Exception:
            tau = np.percentile(variances, 30)  # fallback
            print('percentile instead')

        thresholds.append(tau)
    
    return torch.tensor(thresholds)

z_var = qz_x_corr.variance
thresholds_gmm = compute_gmm_thresholds(z_var)
thresholds_gmm


In [ ]:
gpuae.model.gp.thresholds = thresholds_gmm

In [ ]:
#gpuae.model.backbone.alpha = 1e0
#gpuae.gp.p = 0.1

gpuae.model.backbone.encoder.eval()
gpuae.model.backbone.decoder.eval()
gpuae.model.gp.use_quantile = True
gpuae.model.alpha = .8

gpuae.fit(dataset_train_nan, dataset_val_nan, train_kernel_only = True)  # train the model on the dataset

In [ ]:
from pypots.utils.metrics import calc_mae
import numpy as np

results = gpuae.predict(dataset_val_nan, with_gp = False)
imputation = np.array(results['imputation'])
calc_mae(imputation, dataset_val_nan['X_ori'], (dataset_val_nan['X_ori']==dataset_val_nan['X_ori']))

In [ ]:
import matplotlib.pyplot as plt

i = 1
plt.plot(imputation[i],':');
plt.gca().set_prop_cycle(None)
plt.plot(dataset_val_nan['X_ori'][i],'+');
plt.gca().set_prop_cycle(None)
plt.plot(dataset_val_nan['X'][i],'o');

In [ ]:
import matplotlib.pyplot as plt

for j in gpuae.gp.kernel_params_history.keys():
    hist_j = gpuae.gp.kernel_params_history[j]
    fig, ax = plt.subplots(2, figsize=(18,8), sharex = True)
    for elt in ['a','b','c','d']:
        ax[0].plot(hist_j[elt], label = elt)
    ax[1].semilogy(hist_j['e'])
    fig.legend();
    plt.title(f'Latent dim {j}')
    plt.show();

In [ ]:
torch.save(gpuae.model.backbone.encoder.state_dict().copy(), 'encoder_uae')
torch.save(gpuae.model.backbone.decoder.state_dict().copy(), 'decoder_uae')

In [ ]:
state_dict = torch.load('encoder_uae');
gpuae.model.backbone.encoder.load_state_dict(state_dict);
state_dict = torch.load('decoder_uae');
gpuae.model.backbone.decoder.load_state_dict(state_dict);